In [2]:
import csv
import pandas as pd
from sklearn.cluster import KMeans
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter

Get most common tags and replace the list of tags for each element with the most common tag

In [2]:
# Read the CSV file and extract the tags column
with open('medium_articles.csv', 'r',encoding="utf8") as file:
    reader = csv.DictReader(file)
    tags = [row['tags'] for row in reader]

# Convert each tags list to a Counter object and get the most common tag
most_common_tags = [Counter(eval(tag)).most_common(1)[0][0] for tag in tags]

# Replace the tags column in the original CSV file with the most common tag
with open('medium_articles.csv', 'r',encoding="utf8") as file:
    reader = csv.reader(file)
    headers = next(reader)
    rows = [[*row[:headers.index('tags')], most_common_tags[i], *row[headers.index('tags')+1:]] for i, row in enumerate(reader)]
    
with open('medium_articles_nolist.csv', 'w',encoding="utf8", newline='') as file:
    writer = csv.writer(file)
    writer.writerow(headers)
    writer.writerows(rows)

Now we want to cluster the tags and assign each tag of element to the main cluster

In [10]:
# Load the dataset
data = pd.read_csv('medium_articles_nolist.csv')

# Extract the unique tags
tags = set(data['tags'])
tags = [tag for tag in tags if isinstance(tag, str)]

Vectorize the tags. using TfidfVectorizer from scikit-learn to convert the tags into numerical vectors.

In [11]:
# Vectorize the tags
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(tags)

Apply clustering to group the tags into 6 clusters using kmeans

In [12]:
# Apply KMeans clustering
kmeans = KMeans(n_clusters=6)
kmeans.fit(X)

c:\Users\223079929\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\cluster\_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


KMeans(n_clusters=6)

Assign each tag to its cluster and create a dictionary to map each tag to its main tag.

In [13]:
# Assign each tag to its cluster
clusters = kmeans.labels_

# Create a dictionary to map each tag to its main tag
tag_to_main = {}
for i, tag in enumerate(tags):
    tag_to_main[tag] = f"Main Tag {clusters[i]+1}"

Update the dataset with the main tags. apply the tag_to_main dictionary to the tags column in dataset and create a new column with the main tags.

In [14]:
# Update the dataset with the main tags
data['main_tags'] = data['tags'].apply(lambda x: tag_to_main.get(x, 'unknown') if pd.notnull(x) else 'unknown')
# remove 'unknown' main tag and save the updated dataset to a new csv file
data = data[data['main_tags'] != 'unknown']
data.to_csv('medium_articles_nolist_6tags.csv', index=False)

In [15]:
# Read in the CSV file as a Pandas DataFrame
df = pd.read_csv('medium_articles_nolist_6tags.csv')

# Group the rows by the values in the 'main_tags' column and collect all the tags assigned to each group
tag_groups = df.groupby('main_tags')['tags'].apply(list)

# Loop over the groups and get the most common tag for each group
for group, tags in tag_groups.items():
    tag_counts = Counter(tags)
    top_tag, count = tag_counts.most_common(1)[0]
    print(f"Most common tag for {group}: {top_tag} (count={count})")

Most common tag for Main Tag 1: Money (count=483)
Most common tag for Main Tag 2: Scholarx (count=1)
Most common tag for Main Tag 3: The Daily Pick (count=78)
Most common tag for Main Tag 4: Machine Learning (count=2178)
Most common tag for Main Tag 5: Design (count=595)
Most common tag for Main Tag 6: Guns (count=34)


Remove unecessary columns, switch places remained ones and rename them

In [23]:
# Remove column
data = pd.read_csv('medium_articles_nolist_6tags.csv')
columns_to_remove = ['title', 'url', 'authors', 'timestamp', 'tags']
data = data.drop(columns_to_remove, axis=1)

# Switch columns places
col1 = data['text']
col2 = data['main_tags']
data['text'] = col2
data['main_tags'] = col1

# Rename columns
data = data.rename(columns={'text': 'category', 'main_tags': 'text'})

# Save modified DataFrame to new CSV file
data.to_csv('medium_articles_nolist_6tags_cleared.csv', index=False)

Rename column to its appropriate name from bbc dataset

In [24]:
data = pd.read_csv('medium_articles_nolist_6tags_cleared.csv')
for index, row in data.iterrows():
    if row['category'] == 'Main Tag 5':
        data.iloc[index, 0] = 'marketing'
    if row['category'] == 'Main Tag 4':
        data.iloc[index, 0] = 'tech'
    if row['category'] == 'Main Tag 1':
        data.iloc[index, 0] = 'business'

remove unecessary elements

In [25]:
data = data[data['category'] != 'Main Tag 2']
data = data[data['category'] != 'Main Tag 3']
data = data[data['category'] != 'Main Tag 6']

categories = set(data['category'])
categories
data.to_csv('medium_articles_nolist_6tags_cleared.csv', index=False)

load bbc dataset and rename certain category

In [26]:
data = pd.read_csv('bbc-text.csv')
for index, row in data.iterrows():
    if row['category'] == 'entertainment':
        data.iloc[index, 0] = 'marketing'

In [27]:
categories = set(data['category'])
categories

{'business', 'marketing', 'politics', 'sport', 'tech'}

combine together meium and bbc datasets into one which will be the final dataset for NLP algorithms

In [4]:
# Load the first CSV file into a DataFrame
df1 = pd.read_csv('medium_articles_nolist_6tags_cleared.csv')

# Load the second CSV file into a DataFrame
df2 = pd.read_csv('bbc-text.csv')

# Combine the two DataFrames into a single DataFrame
result = pd.concat([df1, df2])

# Save the combined DataFrame to a new CSV file
result.to_csv('medium_and_bbc.csv', index=False)

In [3]:
data = pd.read_csv('medium_and_bbc.csv')
categories = set(data['category'])
categories

{'business', 'entertainment', 'marketing', 'politics', 'sport', 'tech'}

In [5]:
data[1:4]

,category,text
1,tech,Your Brain On Coronavirus\n\nA guide to the cu...
2,tech,Mind Your Nose\n\nHow smell training can chang...
3,tech,Passionate about the synergy between science a...


In [6]:

category_counts = {}

# Open the CSV file and read the data
with open('medium_and_bbc.csv', 'r', encoding='utf-8') as csv_file:
    reader = csv.DictReader(csv_file)
    for row in reader:
        # Get the category of the current row
        category = row['category']
        
        # Update the count for the current category in the dictionary
        if category in category_counts:
            category_counts[category] += 1
        else:
            category_counts[category] = 1

# Print the category counts
for category, count in category_counts.items():
    print(f'{category}: {count}')


tech: 188506
marketing: 2449
business: 1799
sport: 511
entertainment: 386
politics: 417
